In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs and parse timestamps as UTC datetimes
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

log = pd.read_csv("log.csv")
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
if 'value' in log.columns:
    log['value'] = pd.to_numeric(log['value'], errors='coerce')

trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# Aggregate metric.csv by (cmdb_id, kpi_name): count, mean, median (P50), P95 (global across entire file)
metric_agg = (
    metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        mean='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)

# Top 20 rows sorted by P95 descending
metric_top20 = metric_agg.sort_values('p95', ascending=False).head(20)

# Aggregate log.csv by (cmdb_id, log_name): count and mean(value) if value exists
if 'value' in log.columns:
    log_agg = (
        log
        .groupby(['cmdb_id', 'log_name'])['value']
        .agg(count='count', mean='mean')
        .reset_index()
    )
else:
    log_agg = (
        log
        .groupby(['cmdb_id', 'log_name'])
        .size()
        .reset_index(name='count')
    )
log_agg_head = log_agg.head(20)

# Aggregate trace.csv by (cmdb_id, trace_name): count and mean(value)
trace_agg = (
    trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(count='count', mean='mean')
    .reset_index()
)
trace_agg_head = trace_agg.head(20)

# Full list of unique cmdb_id values across the three files (as a compact Series)
unique_cmdb_ids = pd.Series(
    np.sort(
        pd.unique(
            pd.concat([
                metric['cmdb_id'].dropna(),
                log['cmdb_id'].dropna(),
                trace['cmdb_id'].dropna()
            ]).values
        )
    ),
    name='cmdb_id'
)

# Display compact outputs (DataFrames/Series). Use variable names as output.
metric_top20, log_agg_head, trace_agg_head, unique_cmdb_ids

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry aggregates:\n"
    "- Metrics: Memory usage P95 is highest for catalogue-db (4.33e8), followed by orders, shipping, queue-master and front-end. "
    "Several services show high memory P95 values, indicating system-wide elevated memory usage. "
    "Disk I/O P95 is notable for carts-db and user-db.\n"
    "- Logs: No error counts observed (log.error_count == 0 for all reported services). "
    "log.row_count is high for front-end and queue-master, indicating heavy request/log activity but not error logs.\n"
    "- Traces: No trace aggregates available (trace.csv produced no aggregated rows).\n"
    "- CMDB IDs seen: carts, carts-db, catalogue, catalogue-db, front-end, orders, orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db.\n\n"
    "Overall conclusion: The telemetry points to elevated memory usage (high P95) across multiple components—most strongly in catalogue-db—rather than obvious error logs or trace anomalies. Investigate memory pressure on catalogue-db, orders, shipping and queue-master first, and review disk I/O on carts-db and user-db."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name  count          mean           p50           p95
20       catalogue-db      mem     25  4.328019e+08  4.328120e+08  4.328217e+08
32             orders      mem     25  3.323171e+08  3.323837e+08  3.326000e+08
63           shipping      mem     25  2.983621e+08  2.983506e+08  2.985018e+08
47       queue-master      mem     25  2.546688e+08  2.547671e+08  2.548572e+08
26          front-end      mem     25  1.424805e+08  1.047691e+08  2.523554e+08
3               carts      mem     25  2.102596e+08  2.097368e+08  2.119185e+08
51           rabbitmq      mem     25  9.254063e+07  9.260954e+07  9.261851e+07
8            carts-db      mem     25  8.055983e+07  8.108503e+07  8.199594e+07
74            user-db      mem     25  4.669448e+07  4.709321e+07  4.782442e+07
37          orders-db      mem     25  4.125881e+07  4.138373e+07  4.187756e+07
15          catalogue      mem     25  6.994069e+06  6.107682e+06  9.355087e+06
69               user      mem     25  8.107099e+06  8.112879e+06  8.167124e+06
54  rabbitmq-exporter      mem     25  4.755456e+06  4.755456e+06  4.755456e+06
42            payment      mem     25  4.496136e+06  4.513792e+06  4.525070e+06
58         session-db      mem     25  2.904015e+06  2.893551e+06  2.935369e+06
7            carts-db   diskio     25  1.530608e+06  1.906240e+06  2.053568e+06
73            user-db   diskio     25  1.095115e+06  1.296428e+06  1.566922e+06
36          orders-db   diskio     25  6.294433e+05  7.290475e+05  8.496449e+05
46       queue-master   diskio     25  3.485002e+04  3.636606e+04  5.389049e+04
57         session-db   diskio     25  4.002387e+03  0.000000e+00  1.997194e+04,          cmdb_id         log_name  count         mean
0          carts  log.error_count     17     0.000000
1          carts    log.row_count     17    70.588235
2       carts-db  log.error_count     23     0.000000
3       carts-db    log.row_count     23     1.086957
4      catalogue  log.error_count     20     0.000000
5      catalogue    log.row_count     20   133.900000
6      front-end  log.error_count     24     0.000000
7      front-end    log.row_count     24  1412.833333
8         orders  log.error_count     19     0.000000
9         orders    log.row_count     19   104.315789
10     orders-db  log.error_count      1     0.000000
11     orders-db    log.row_count      1     5.000000
12       payment  log.error_count     24     0.000000
13       payment    log.row_count     24    81.291667
14  queue-master  log.error_count     19     0.000000
15  queue-master    log.row_count     19   469.421053
16    session-db  log.error_count      5     0.000000
17    session-db    log.row_count      5     5.000000
18      shipping  log.error_count     19     0.000000
19      shipping    log.row_count     19    52.157895, Empty DataFrame
Columns: [cmdb_id, trace_name, count, mean]
Index: [], 0                 carts
1              carts-db
2             catalogue
3          catalogue-db
4             front-end
5                orders
6             orders-db
7               payment
8          queue-master
9              rabbitmq
10    rabbitmq-exporter
11           session-db
12             shipping
13                 user
14              user-db
Name: cmdb_id, dtype: object)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse previously loaded 'metric' and 'metric_agg' from kernel memory.
# Define incident window in UTC
start = pd.Timestamp("2024-01-20 01:21:01", tz="UTC")
end = pd.Timestamp("2024-01-20 01:51:01", tz="UTC")

# Prepare global P95 thresholds (already computed globally in metric_agg)
thresholds = metric_agg[['cmdb_id', 'kpi_name', 'p95']].rename(columns={'p95': 'global_p95'})

# Filter metric rows to the incident window (inclusive)
metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

# Merge global P95 into windowed data
metric_w_th = metric_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Drop rows without a global threshold (cannot evaluate anomaly)
metric_w_th = metric_w_th.dropna(subset=['global_p95'])

# Flag anomalies where value >= global P95
metric_w_th['is_anom'] = metric_w_th['value'] >= metric_w_th['global_p95']

# For earliest anomaly timestamp, keep timestamp when is_anom else NaT
metric_w_th['anom_ts'] = metric_w_th['timestamp'].where(metric_w_th['is_anom'], pd.NaT)

# Aggregate per (cmdb_id, kpi_name)
agg = (
    metric_w_th
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        total_points_in_window = ('value', 'count'),
        anomaly_count = ('is_anom', 'sum'),
        earliest_anomaly_timestamp = ('anom_ts', 'min'),
        max_value_in_window = ('value', 'max'),
        global_P95 = ('global_p95', 'first')
    )
    .reset_index()
)

# Compute severity_ratio safely (handle global_P95 == 0)
agg['severity_ratio'] = agg.apply(
    lambda r: np.inf if r['global_P95'] == 0 else r['max_value_in_window'] / r['global_P95'],
    axis=1
)

# Only include rows with anomaly_count > 0
anomalies = agg[agg['anomaly_count'] > 0].copy()

# Sort by severity_ratio descending and return top 20, keep columns compact and ordered
anomalies_top20 = (
    anomalies
    .sort_values('severity_ratio', ascending=False)
    [['cmdb_id', 'kpi_name', 'total_points_in_window', 'anomaly_count',
      'earliest_anomaly_timestamp', 'max_value_in_window', 'global_P95', 'severity_ratio']]
    .head(20)
)

anomalies_top20

```
Out[2]:
```
```python
summary = (
    "Summary of metric-window anomalies (2024-01-20 01:21:01 to 01:51:01 UTC):\n"
    "- Most severe items: disk I/O series for catalogue, catalogue-db and rabbitmq show anomalies for every point in-window. Their global P95 is recorded as 0, producing infinite severity_ratio — investigate these disk I/O measurements and the global threshold calculation immediately.\n"
    "- High severity (severity_ratio > 1.3): catalogue CPU (severity ~2.47) and queue-master diskio (1.54). These indicate values well above the global P95.\n"
    "- Moderate severity (1.1–1.7): front-end error, catalogue error, carts CPU, front-end socket, queue-master CPU, shipping CPU, and several latency metrics across catalogue/orders/payment show modest exceedances of global P95.\n"
    "- Timing: earliest clustered anomalies occur around 2024-01-20 01:24:00 UTC (many diskio/CPU anomalies) and 01:37–01:40 UTC (error/latency/socket anomalies).\n"
    "Conclusion / next steps: prioritize investigating disk I/O for catalogue, catalogue-db and rabbitmq (check metric collection and why global P95==0), then check CPU pressure on catalogue, queue-master and front-end, and investigate socket/latency anomalies affecting front-end, catalogue and orders."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_P95  severity_ratio
11          catalogue      diskio                      14             14  2024-01-20 01:24:00+00:00         0.000000e+00  0.000000e+00             inf
19       catalogue-db      diskio                      25             25  2024-01-20 01:24:00+00:00         0.000000e+00  0.000000e+00             inf
50           rabbitmq      diskio                      25             25  2024-01-20 01:24:00+00:00         0.000000e+00  0.000000e+00             inf
10          catalogue         cpu                      25              2  2024-01-20 01:24:00+00:00         5.520983e-01  2.237552e-01        2.467421
23          front-end       error                      25              2  2024-01-20 01:37:00+00:00         1.540083e+00  9.200000e-01        1.674004
46       queue-master      diskio                      25              2  2024-01-20 01:24:00+00:00         8.287603e+04  5.389049e+04        1.537860
12          catalogue       error                      25              2  2024-01-20 01:37:00+00:00         1.500000e-01  1.050133e-01        1.428390
0               carts         cpu                      25              2  2024-01-20 01:24:00+00:00         2.659148e+00  1.926449e+00        1.380337
27          front-end      socket                      25              2  2024-01-20 01:40:00+00:00         3.361667e+01  2.741333e+01        1.226289
45       queue-master         cpu                      25              2  2024-01-20 01:29:00+00:00         1.788716e+00  1.558095e+00        1.148015
60           shipping         cpu                      25              2  2024-01-20 01:31:00+00:00         8.859463e-01  7.730790e-01        1.145997
13          catalogue  latency-50                      21              2  2024-01-20 01:40:00+00:00         2.257421e-01  2.066778e-01        1.092242
62           shipping  latency-90                      22              2  2024-01-20 01:40:00+00:00         5.416667e-03  5.016378e-03        1.079796
16          catalogue      socket                      25              2  2024-01-20 01:40:00+00:00         3.010000e+01  2.806000e+01        1.072701
40            payment  latency-50                      22              2  2024-01-20 01:37:00+00:00         3.000000e-03  2.803264e-03        1.070181
25          front-end  latency-90                      25              2  2024-01-20 01:47:00+00:00         3.281915e-01  3.073123e-01        1.067941
53  rabbitmq-exporter         cpu                      25              2  2024-01-20 01:38:00+00:00         1.461113e-04  1.372678e-04        1.064425
73            user-db      diskio                      25              2  2024-01-20 01:29:00+00:00         1.662097e+06  1.566922e+06        1.060740
30             orders  latency-50                      22              2  2024-01-20 01:32:00+00:00         3.750000e-02  3.578830e-02        1.047829
33             orders      socket                      25              3  2024-01-20 01:25:00+00:00         2.075000e+01  1.983333e+01        1.046218```
```

In [ ]:
# In[3]:

# Reuse existing 'metric' DataFrame from kernel memory.

import numpy as np
import pandas as pd

# Define targets
diskio_ids = ['catalogue', 'catalogue-db', 'rabbitmq']

# Filter full-file series for diskio for the selected components
diskio_df = metric[(metric['kpi_name'] == 'diskio') & (metric['cmdb_id'].isin(diskio_ids))].copy()

# Global aggregates for diskio (one row per component)
diskio_aggs = (
    diskio_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        min='min',
        max='max',
        mean='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)

# Up to 20 sample raw diskio rows across these components sorted by timestamp (timestamp already UTC datetime)
diskio_samples = (
    diskio_df
    .sort_values('timestamp')
    .loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value']]
    .head(20)
    .reset_index(drop=True)
)

# For cpu and cmdb_id == 'catalogue'
cpu_cat_df = metric[(metric['kpi_name'] == 'cpu') & (metric['cmdb_id'] == 'catalogue')].copy()

cpu_catalogue_agg = (
    cpu_cat_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        min='min',
        max='max',
        mean='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)

cpu_catalogue_samples = (
    cpu_cat_df
    .sort_values('timestamp')
    .loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value']]
    .head(20)
    .reset_index(drop=True)
)

# Display compact results
diskio_aggs, diskio_samples, cpu_catalogue_agg, cpu_catalogue_samples

```
Out[3]:
```
summary = (
    "Results summary (full-file):\n"
    "- Disk I/O (catalogue, catalogue-db, rabbitmq):\n"
    "  * All aggregated statistics are 0.0 (count: catalogue=14, catalogue-db=25, rabbitmq=25; min=max=mean=p50=p95=0.0).\n"
    "  * Sample raw rows (up to 20) show diskio value 0.0 at every timestamp (timestamps around 2024-01-20 01:24–01:30 UTC).\n"
    "  * Implication: the global P95 for these diskio series is 0 — this explains the infinite severity_ratio seen earlier and suggests missing/non-varying diskio data or metric-collection issues, not necessarily true high disk I/O.\n\n"
    "- CPU for catalogue:\n"
    "  * Aggregates (count=25): min=0.001901, max=0.552098, mean≈0.1658, p50≈0.1908, p95≈0.2238.\n"
    "  * Sample raw rows (up to 20) show a peak at 2024-01-20 01:24:00 UTC (0.552098) and typical values ~0.19–0.21 for many minutes, with a few very small readings later.\n"
    "  * Implication: catalogue CPU occasionally exceeded its global P95 (notably at 01:24), consistent with the CPU anomaly flagged earlier.\n\n"
    "Actionable recommendation: investigate why diskio metrics for catalogue/catalogue-db/rabbitmq are all zeros (metric collection, exporter config, or labeling), and separately investigate the CPU spike on catalogue around 2024-01-20 01:24 UTC."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id kpi_name  count  min  max  mean  p50  p95
0     catalogue   diskio     14  0.0  0.0   0.0  0.0  0.0
1  catalogue-db   diskio     25  0.0  0.0   0.0  0.0  0.0
2      rabbitmq   diskio     25  0.0  0.0   0.0  0.0  0.0,                    timestamp       cmdb_id kpi_name  value
0  2024-01-20 01:24:00+00:00     catalogue   diskio    0.0
1  2024-01-20 01:24:00+00:00  catalogue-db   diskio    0.0
2  2024-01-20 01:24:00+00:00      rabbitmq   diskio    0.0
3  2024-01-20 01:25:00+00:00     catalogue   diskio    0.0
4  2024-01-20 01:25:00+00:00  catalogue-db   diskio    0.0
5  2024-01-20 01:25:00+00:00      rabbitmq   diskio    0.0
6  2024-01-20 01:26:00+00:00     catalogue   diskio    0.0
7  2024-01-20 01:26:00+00:00  catalogue-db   diskio    0.0
8  2024-01-20 01:26:00+00:00      rabbitmq   diskio    0.0
9  2024-01-20 01:27:00+00:00     catalogue   diskio    0.0
10 2024-01-20 01:27:00+00:00  catalogue-db   diskio    0.0
11 2024-01-20 01:27:00+00:00      rabbitmq   diskio    0.0
12 2024-01-20 01:28:00+00:00     catalogue   diskio    0.0
13 2024-01-20 01:28:00+00:00  catalogue-db   diskio    0.0
14 2024-01-20 01:28:00+00:00      rabbitmq   diskio    0.0
15 2024-01-20 01:29:00+00:00      rabbitmq   diskio    0.0
16 2024-01-20 01:29:00+00:00     catalogue   diskio    0.0
17 2024-01-20 01:29:00+00:00  catalogue-db   diskio    0.0
18 2024-01-20 01:30:00+00:00     catalogue   diskio    0.0
19 2024-01-20 01:30:00+00:00  catalogue-db   diskio    0.0,      cmdb_id kpi_name  count       min       max      mean       p50       p95
0  catalogue      cpu     25  0.001901  0.552098  0.165778  0.190786  0.223755,                    timestamp    cmdb_id kpi_name     value
0  2024-01-20 01:24:00+00:00  catalogue      cpu  0.552098
1  2024-01-20 01:25:00+00:00  catalogue      cpu  0.190786
2  2024-01-20 01:26:00+00:00  catalogue      cpu  0.200890
3  2024-01-20 01:27:00+00:00  catalogue      cpu  0.214212
4  2024-01-20 01:28:00+00:00  catalogue      cpu  0.199611
5  2024-01-20 01:29:00+00:00  catalogue      cpu  0.204087
6  2024-01-20 01:30:00+00:00  catalogue      cpu  0.210184
7  2024-01-20 01:31:00+00:00  catalogue      cpu  0.190014
8  2024-01-20 01:32:00+00:00  catalogue      cpu  0.199232
9  2024-01-20 01:33:00+00:00  catalogue      cpu  0.203580
10 2024-01-20 01:34:00+00:00  catalogue      cpu  0.213296
11 2024-01-20 01:35:00+00:00  catalogue      cpu  0.214081
12 2024-01-20 01:36:00+00:00  catalogue      cpu  0.226141
13 2024-01-20 01:37:00+00:00  catalogue      cpu  0.105094
14 2024-01-20 01:38:00+00:00  catalogue      cpu  0.002017
15 2024-01-20 01:39:00+00:00  catalogue      cpu  0.001937
16 2024-01-20 01:40:00+00:00  catalogue      cpu  0.154001
17 2024-01-20 01:41:00+00:00  catalogue      cpu  0.180558
18 2024-01-20 01:42:00+00:00  catalogue      cpu  0.142424
19 2024-01-20 01:43:00+00:00  catalogue      cpu  0.070636)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse metric DataFrame from kernel memory and ensure timestamp is datetime UTC
# (Assuming metric already loaded and parsed in previous steps)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.Timestamp("2024-01-20 01:21:01", tz="UTC")
end = pd.Timestamp("2024-01-20 01:51:01", tz="UTC")

# Candidate list
candidates = [
    ("catalogue", "cpu"),
    ("catalogue", "error"),
    ("front-end", "error"),
    ("front-end", "socket"),
    ("queue-master", "cpu"),
    ("queue-master", "diskio"),
    ("orders", "socket"),
]

# Ensure global P95 thresholds are available (compute if not present)
if 'thresholds' in globals():
    thresholds_df = thresholds.copy()
else:
    metric_agg = (
        metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95))
        .reset_index()
    )
    thresholds_df = metric_agg.rename(columns={'p95': 'global_p95'})

# Rename threshold column consistently
if 'global_p95' not in thresholds_df.columns:
    thresholds_df = thresholds_df.rename(columns={'p95': 'global_p95'})

# Filter metric to the incident window and to candidate pairs
cand_set = set(candidates)
window_df = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()
window_df = window_df[window_df.apply(lambda r: (r['cmdb_id'], r['kpi_name']) in cand_set, axis=1)].copy()

# Merge global thresholds (calculated across full file) into the windowed data
window_df = window_df.merge(thresholds_df[['cmdb_id', 'kpi_name', 'global_p95']],
                            on=['cmdb_id', 'kpi_name'], how='left')

# Drop rows lacking a global threshold (cannot evaluate anomaly)
window_df = window_df.dropna(subset=['global_p95']).copy()

# Flag anomalies
window_df['is_anom'] = window_df['value'] >= window_df['global_p95']

# Prepare containers for segments and summaries
segments = []
summary_rows = []

# Process each candidate group
for (cmdb_id, kpi_name) in candidates:
    grp = window_df[(window_df['cmdb_id'] == cmdb_id) & (window_df['kpi_name'] == kpi_name)].copy()
    if grp.empty:
        # add summary with zeros/NaT
        summary_rows.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'total_anomaly_points_in_window': 0,
            'earliest_anomaly_timestamp': pd.NaT
        })
        continue

    # Summary totals
    total_anom = int(grp['is_anom'].sum())
    earliest_ts = grp.loc[grp['is_anom'], 'timestamp'].min() if total_anom > 0 else pd.NaT
    summary_rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'total_anomaly_points_in_window': total_anom,
        'earliest_anomaly_timestamp': earliest_ts
    })

    if total_anom == 0:
        continue

    # Work on anomaly rows to find consecutive segments (per-minute consecutive)
    anom = grp[grp['is_anom']].sort_values('timestamp').copy()
    # compute minute diff; first diff treated as new segment
    anom['diff_min'] = anom['timestamp'].diff().dt.total_seconds().div(60).fillna(0)
    anom['segment_id'] = (anom['diff_min'] != 1).cumsum()

    # Aggregate segments
    seg_aggs = (
        anom
        .groupby('segment_id')
        .agg(
            cmdb_id=('cmdb_id', 'first'),
            kpi_name=('kpi_name', 'first'),
            segment_start_utc=('timestamp', 'min'),
            segment_end_utc=('timestamp', 'max'),
            segment_length_minutes=('timestamp', 'count'),
            anomaly_count_in_segment=('timestamp', 'count'),
            max_value_in_segment=('value', 'max'),
            global_P95=('global_p95', 'first')
        )
        .reset_index(drop=True)
    )

    # Keep only segments with length >= 2 minutes
    seg_aggs = seg_aggs[seg_aggs['segment_length_minutes'] >= 2].copy()
    if seg_aggs.empty:
        continue

    # Compute severity ratio safely
    def safe_ratio(row):
        gp = row['global_P95']
        if gp == 0:
            return np.inf
        return row['max_value_in_segment'] / gp

    seg_aggs['severity_ratio'] = seg_aggs.apply(safe_ratio, axis=1)

    # Limit to up to 5 segments per component-KPI
    seg_aggs = seg_aggs.sort_values('segment_start_utc').head(5)

    # Collect
    for _, r in seg_aggs.iterrows():
        segments.append({
            'cmdb_id': r['cmdb_id'],
            'kpi_name': r['kpi_name'],
            'segment_start_utc': r['segment_start_utc'],
            'segment_end_utc': r['segment_end_utc'],
            'segment_length_minutes': int(r['segment_length_minutes']),
            'anomaly_count_in_segment': int(r['anomaly_count_in_segment']),
            'max_value_in_segment': r['max_value_in_segment'],
            'global_P95': r['global_P95'],
            'severity_ratio': r['severity_ratio']
        })

# Build DataFrames
segments_df = pd.DataFrame(segments)
# Sort segments by severity_ratio descending and limit to 20 rows
if not segments_df.empty:
    segments_top20 = segments_df.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)
else:
    segments_top20 = segments_df  # empty

summary_df = pd.DataFrame(summary_rows)
# compact ordering for summary
summary_df = summary_df[['cmdb_id', 'kpi_name', 'total_anomaly_points_in_window', 'earliest_anomaly_timestamp']]

# Final outputs: segments_top20 (<=20 rows) and summary_df (one row per candidate)
segments_top20, summary_df.head(len(candidates))

```
Out[4]:
```
summary = (
    "Summary for candidate component-KPIs (window 2024-01-20 01:21:01 to 01:51:01 UTC):\n"
    "- No qualifying consecutive anomaly segments of length >= 2 minutes were found for the listed component-KPIs.\n"
    "- Observed anomaly points (count and earliest anomaly timestamp):\n"
    "  * catalogue|cpu: 2 anomalies, earliest 2024-01-20 01:24:00 UTC\n"
    "  * catalogue|error: 2 anomalies, earliest 2024-01-20 01:37:00 UTC\n"
    "  * front-end|error: 2 anomalies, earliest 2024-01-20 01:37:00 UTC\n"
    "  * front-end|socket: 2 anomalies, earliest 2024-01-20 01:40:00 UTC\n"
    "  * queue-master|cpu: 2 anomalies, earliest 2024-01-20 01:29:00 UTC\n"
    "  * queue-master|diskio: 2 anomalies, earliest 2024-01-20 01:24:00 UTC\n"
    "  * orders|socket: 3 anomalies, earliest 2024-01-20 01:25:00 UTC\n"
    "- Interpretation and next steps: anomalies exist but are not consecutive per-minute runs (they appear as isolated points). Focus investigation on the listed timestamps and components—particularly orders|socket (3 points) and catalogue|cpu—correlate with logs/traces and check metric sampling/collection to rule out irregularities."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: []
Index: [],         cmdb_id kpi_name  total_anomaly_points_in_window earliest_anomaly_timestamp
0     catalogue      cpu                               2  2024-01-20 01:24:00+00:00
1     catalogue    error                               2  2024-01-20 01:37:00+00:00
2     front-end    error                               2  2024-01-20 01:37:00+00:00
3     front-end   socket                               2  2024-01-20 01:40:00+00:00
4  queue-master      cpu                               2  2024-01-20 01:29:00+00:00
5  queue-master   diskio                               2  2024-01-20 01:24:00+00:00
6        orders   socket                               3  2024-01-20 01:25:00+00:00)```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse existing 'log' DataFrame if present; otherwise load it
try:
    log
except NameError:
    log = pd.read_csv("log.csv")
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed and value numeric if exists
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
if 'value' in log.columns:
    log['value'] = pd.to_numeric(log['value'], errors='coerce')

# Incident window (UTC)
start = pd.Timestamp("2024-01-20 01:21:01", tz="UTC")
end = pd.Timestamp("2024-01-20 01:51:01", tz="UTC")

# Filter log to window (inclusive)
log_window = log[(log['timestamp'] >= start) & (log['timestamp'] <= end)].copy()

# Aggregate by (cmdb_id, log_name)
if 'value' in log_window.columns:
    log_agg = (
        log_window
        .groupby(['cmdb_id', 'log_name'], dropna=False)
        .agg(
            count_rows_in_window = ('value', 'count'),
            mean_value = ('value', 'mean')
        )
        .reset_index()
    )
else:
    log_agg = (
        log_window
        .groupby(['cmdb_id', 'log_name'], dropna=False)
        .size()
        .reset_index(name='count_rows_in_window')
    )

# Sort by count descending and keep up to 20 rows
log_agg_head = log_agg.sort_values('count_rows_in_window', ascending=False).head(20)

# Load and process error_logs.csv
error_logs = pd.read_csv("error_logs.csv")
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Filter error logs to the same incident window
err_window = error_logs[(error_logs['timestamp'] >= start) & (error_logs['timestamp'] <= end)].copy()

# Per-cmdb_id aggregates: count_of_messages and earliest_message_timestamp
err_agg = (
    err_window
    .groupby('cmdb_id', dropna=False)
    .agg(
        count_of_messages = ('message', 'count'),
        earliest_message_timestamp = ('timestamp', 'min')
    )
    .reset_index()
    .sort_values('count_of_messages', ascending=False)
)

# Up to 20 sample messages sorted by timestamp ascending (compact)
err_samples = err_window.sort_values('timestamp').loc[:, ['timestamp', 'cmdb_id', 'message']].head(20).reset_index(drop=True)

# Display compact outputs
log_agg_head, err_agg, err_samples

```
Out[5]:
```
Summary (window 2024-01-20 01:21:01 → 2024-01-20 01:51:01 UTC):

- Log aggregates:
  - No services reported non-zero error counts in this window: log.error_count == 0 for all reported cmdb_ids.
  - Highest log activity (log.row_count mean / counts):
    - front-end: 24 rows, mean row_count ≈ 1412.83
    - user: 24 rows, mean row_count ≈ 415.54
    - queue-master: 19 rows, mean row_count ≈ 469.42
    - catalogue: 20 rows, mean row_count ≈ 133.90
    - orders: 19 rows, mean row_count ≈ 104.32
    - payment: 24 rows, mean row_count ≈ 81.29
    - shipping: 19 rows, mean row_count ≈ 52.16
    - carts: 17 rows, mean row_count ≈ 70.59
    - session-db: 5 rows, mean row_count = 5.00
  - Interpretation: front-end and user show the heaviest log/traffic volumes; despite high row counts, error_count fields are zero.

- Error logs (error_logs.csv) in the same window:
  - No error log messages found (zero rows), so per-cmdb_id counts, earliest timestamps, and sample messages are empty.

- Recommended next steps:
  - Investigate high activity components (front-end, user, queue-master) for load/latency effects even though log.error_count is zero.
  - Correlate with metrics/traces (CPU, latency, socket) around the timestamps flagged earlier to find operational impact.
  - Verify error_logs ingestion/filters to ensure none were dropped.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  count_rows_in_window   mean_value
20          user  log.error_count                    24     0.000000
21          user    log.row_count                    24   415.541667
13       payment    log.row_count                    24    81.291667
12       payment  log.error_count                    24     0.000000
7      front-end    log.row_count                    24  1412.833333
6      front-end  log.error_count                    24     0.000000
2       carts-db  log.error_count                    23     0.000000
3       carts-db    log.row_count                    23     1.086957
5      catalogue    log.row_count                    20   133.900000
4      catalogue  log.error_count                    20     0.000000
19      shipping    log.row_count                    19    52.157895
18      shipping  log.error_count                    19     0.000000
9         orders    log.row_count                    19   104.315789
8         orders  log.error_count                    19     0.000000
14  queue-master  log.error_count                    19     0.000000
15  queue-master    log.row_count                    19   469.421053
0          carts  log.error_count                    17     0.000000
1          carts    log.row_count                    17    70.588235
17    session-db    log.row_count                     5     5.000000
16    session-db  log.error_count                     5     0.000000, Empty DataFrame
Columns: [cmdb_id, count_of_messages, earliest_message_timestamp]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse metric, log, trace DataFrames from kernel memory (they were loaded earlier).
# Ensure timestamps are datetimes in UTC
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
try:
    trace
    trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
except NameError:
    trace = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])

# Incident window (UTC)
start = pd.Timestamp("2024-01-20 01:21:01", tz="UTC")
end = pd.Timestamp("2024-01-20 01:51:01", tz="UTC")

# Build per-minute timeline (use floored minutes to align with telemetry)
minutes = pd.date_range(start.floor('min'), end.floor('min'), freq='T', tz='UTC')
timeline_df = pd.DataFrame({'timestamp': minutes})

# Helper to get series from metric by cmdb_id and kpi_name condition
def series_from_metric(cmdb_id, kpi_filter):
    m = metric[(metric['cmdb_id'] == cmdb_id) & (metric['kpi_name'].str.contains(kpi_filter, na=False))]
    if m.empty:
        return pd.DataFrame(columns=['timestamp', 'value'])
    s = m.loc[:, ['timestamp', 'value']].drop_duplicates(subset=['timestamp']).sort_values('timestamp')
    return s

# 1) catalogue|cpu
cat_cpu = series_from_metric('catalogue', r'^cpu$')  # exact 'cpu'
cat_cpu = cat_cpu[(cat_cpu['timestamp'] >= start) & (cat_cpu['timestamp'] <= end)]
cat_cpu = cat_cpu.rename(columns={'value': 'catalogue_cpu'})

# 2) orders|socket  (kpi_name containing 'socket')
orders_socket = series_from_metric('orders', r'socket')
orders_socket = orders_socket[(orders_socket['timestamp'] >= start) & (orders_socket['timestamp'] <= end)]
orders_socket = orders_socket.rename(columns={'value': 'orders_socket'})

# 3) front-end|latency-90: prefer metric, else trace, else log (search for 'latency-90' or 'latency-90' substring)
fe_latency = series_from_metric('front-end', r'latency-90')
if fe_latency.empty:
    # try trace
    fe_t = trace[(trace['cmdb_id'] == 'front-end') & (trace['trace_name'].str.contains('latency-90', na=False))]
    if not fe_t.empty:
        fe_latency = fe_t.loc[:, ['timestamp', 'value']].drop_duplicates(subset=['timestamp']).sort_values('timestamp')
    else:
        # try more generic 'latency' names in metric/trace
        fe_latency = series_from_metric('front-end', r'latency')
        if fe_latency.empty:
            fe_t2 = trace[(trace['cmdb_id'] == 'front-end') & (trace['trace_name'].str.contains('latency', na=False))]
            if not fe_t2.empty:
                fe_latency = fe_t2.loc[:, ['timestamp', 'value']].drop_duplicates(subset=['timestamp']).sort_values('timestamp')

fe_latency = fe_latency[(fe_latency['timestamp'] >= start) & (fe_latency['timestamp'] <= end)]
fe_latency = fe_latency.rename(columns={'value': 'front_end_latency90'})

# 4) front-end log.row_count
fe_log = log[(log['cmdb_id'] == 'front-end') & (log['log_name'] == 'log.row_count')].copy()
fe_log = fe_log[(fe_log['timestamp'] >= start) & (fe_log['timestamp'] <= end)]
fe_log = fe_log.loc[:, ['timestamp', 'value']].drop_duplicates(subset=['timestamp']).sort_values('timestamp').rename(columns={'value': 'front_end_log_row_count'})

# Merge all series onto the full-minute timeline
timeline_df = timeline_df.merge(cat_cpu, on='timestamp', how='left')
timeline_df = timeline_df.merge(orders_socket, on='timestamp', how='left')
timeline_df = timeline_df.merge(fe_latency, on='timestamp', how='left')
timeline_df = timeline_df.merge(fe_log, on='timestamp', how='left')

# Keep only requested columns and at most 50 rows (there will be <= 31 minutes in this window)
output_cols = ['timestamp', 'catalogue_cpu', 'orders_socket', 'front_end_latency90', 'front_end_log_row_count']
timeline_result = timeline_df.loc[:, output_cols].head(50)

timeline_result

```
Out[6]:
```
```python
summary = (
    "Per-minute timeline (2024-01-20 01:21–01:51 UTC) — key observations:\n"
    "- Time coverage: data is sparse before 01:24 and after 01:48; most values appear between 01:24 and 01:48.\n"
    "- catalogue|cpu: a clear spike at 01:24 (0.552098), with typical values ~0.19–0.21 afterwards — the 01:24 value is the window peak and exceeds the global P95.\n"
    "- orders|socket: elevated values around 01:24–01:26 (≈19–21), then gradually lower but still present through ~01:47; indicates socket activity concentrated early in the window.\n"
    "- front-end|latency-90: modest at 01:24–01:25 (~0.08), then rises later to ~0.32 by 01:47–01:48 — two latency clusters (early low, later higher).\n"
    "- front-end log.row_count: very high at 01:24–01:25 (~1,950 rows), then declines to ~1,092 by 01:47 and is missing afterward — front-end saw peak traffic early in the window.\n\n"
    "Interpretation: peak catalogue CPU and high orders socket activity coincide with the front-end traffic surge around 01:24–01:25. Latency issues appear later (~01:47–01:48) when front-end load has decreased, suggesting separate periods of stress to investigate (early traffic spike vs. later latency increase)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp  catalogue_cpu  orders_socket  front_end_latency90  front_end_log_row_count
0  2024-01-20 01:21:00+00:00            NaN            NaN                  NaN                      NaN
1  2024-01-20 01:22:00+00:00            NaN            NaN                  NaN                      NaN
2  2024-01-20 01:23:00+00:00            NaN            NaN                  NaN                      NaN
3  2024-01-20 01:24:00+00:00       0.552098      19.338983             0.085366                   1952.0
4  2024-01-20 01:25:00+00:00       0.190786      20.750000             0.080905                   1949.0
..                       ...            ...            ...                  ...                      ...
26 2024-01-20 01:47:00+00:00       0.201508      17.466667             0.319794                   1092.0
27 2024-01-20 01:48:00+00:00       0.105209      13.000000             0.328191                      NaN
28 2024-01-20 01:49:00+00:00            NaN            NaN                  NaN                      NaN
29 2024-01-20 01:50:00+00:00            NaN            NaN                  NaN                      NaN
30 2024-01-20 01:51:00+00:00            NaN            NaN                  NaN                      NaN

[31 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```